# Clustering Analysis with Geospatial Data

This notebook analyzes and compares clustering results using different approaches for incorporating polygon information into HDBSCAN clustering. Three scenarios are examined:

1. **Scenario A: Only Original Points** - clustering using point geometries alone
2. **Scenario B: Points + Frequent Areas** - combining points with derived polygon-area data  
3. **Scenario C: Points + Polygon Centroids** - combining points with centroid data

Each method influences cluster size, density parameters, and spatial coverage differently.

## Key Functions Overview

### find_most_frequent_polygon_area(polygons, grid_size_meters=100)
Identifies geographic regions where input polygons overlap most frequently by creating a grid and counting intersections.

### optimize_and_cluster_geometries(geometries, true_lat, n_trials)
Uses HDBSCAN with Optuna optimization to find optimal clustering parameters (min_samples, eps_meters) and returns cluster polygons.

### display_geospatial_dataset(geometries, true_lat, true_lon, cluster_layers)
Visualizes geospatial data and clustering results on interactive Folium maps with toggleable layers.

In [1]:
# Check if running in Google Colab
try:
    from google.colab import drive
    print("Running in Google Colab. Mounting Google Drive...")
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print("Not running in Google Colab. Skipping Google Drive mount.")
    IN_COLAB = False

Not running in Google Colab. Skipping Google Drive mount.


In [2]:
!pip install optuna
!pip install hdbscan
!pip install folium
!pip install shapely
import os
import sys

# Add the parent directory of ng to the Python path for package imports (only needed in Colab)
if IN_COLAB:
    module_path = '/content/drive/MyDrive/'
    if module_path not in sys.path:
        sys.path.append(module_path)

# Basic imports still needed in the main notebook
import hdbscan
import numpy as np
import optuna
import folium
import json
import datetime
from shapely.geometry import mapping # Only mapping, as Point, Polygon, MultiPoint, box are imported within the utils
import importlib

# Define the constant, consistent with previous usage
DEGREES_PER_METER_LAT = 0.0000089

# Import functions from the correct utils package depending on environment
if IN_COLAB:
    # In Colab, use the mounted Drive copy of the ng package
    import ng.utils.haversine_distance as haversine_distance_module
    import ng.utils.extract_points_from_geometries as extract_points_module
    import ng.utils.calculate_median_cluster_radius_meters as median_radius_module
    import ng.utils.cluster_points_and_get_all_cluster_polygons as cluster_polygons_module
    import ng.utils.optimize_and_cluster_geometries as optimize_cluster_module
    import ng.utils.generate_geospatial_dataset as generate_dataset_module
    import ng.utils.display_geospatial_dataset as display_dataset_module
    import ng.utils.find_most_frequent_polygon_area as frequent_area_module
    import ng.utils.convert_polygon_to_points as convert_polygon_module
    import ng.utils.polygons_to_random_points as polygons_to_random_points_module
    import ng.utils.extract_points_from_geometries as extract_points_from_geometries_module
else:
    # When running locally, import from the local utils package
    import utils.haversine_distance as haversine_distance_module
    import utils.extract_points_from_geometries as extract_points_module
    import utils.calculate_median_cluster_radius_meters as median_radius_module
    import utils.cluster_points_and_get_all_cluster_polygons as cluster_polygons_module
    import utils.optimize_and_cluster_geometries as optimize_cluster_module
    import utils.generate_geospatial_dataset as generate_dataset_module
    import utils.display_geospatial_dataset as display_dataset_module
    import utils.find_most_frequent_polygon_area as frequent_area_module
    import utils.convert_polygon_to_points as convert_polygon_module
    import utils.polygons_to_random_points as polygons_to_random_points_module
    import utils.extract_points_from_geometries as extract_points_from_geometries_module

# Reload modules so edits are picked up without restarting the kernel
importlib.reload(haversine_distance_module)
importlib.reload(extract_points_module)
importlib.reload(median_radius_module)
importlib.reload(cluster_polygons_module)
importlib.reload(optimize_cluster_module)
importlib.reload(generate_dataset_module)
importlib.reload(display_dataset_module)
importlib.reload(frequent_area_module)
importlib.reload(convert_polygon_module)
importlib.reload(polygons_to_random_points_module)
importlib.reload(extract_points_from_geometries_module)

print("All functions imported from ng.utils.")

c:\ng\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All functions imported from ng.utils.


---
## Step 1: Generate Geospatial Dataset

In [3]:
new_geometries, new_true_lat, new_true_lon = generate_dataset_module.generate_geospatial_dataset()

print(f"Generated new dataset with {len(new_geometries)} geometries around Lat: {new_true_lat:.4f}, Lon: {new_true_lon:.4f}")

Generated True Location: (30.4527, 34.3552)
Calculated Degrees per Meter (Lat): 0.00000890
Calculated Degrees per Meter (Lon): 0.00001032
Generated 100 'most' points (0-20m bell curve).
Generated 20 'diverse' polygons (16 containing true location, 4 non-intersecting).
Generated new dataset with 120 geometries around Lat: 30.4527, Lon: 34.3552


In [ ]:
from shapely.geometry import Point, Polygon, MultiPoint, box

original_points = extract_points_from_geometries_module.extract_points_from_geometries(new_geometries)
print(f"Extracted {len(original_points)} points from the geometries.")

original_polygons = [geom for geom in new_geometries if isinstance(geom, Polygon)]
print(f"Found {len(original_polygons)} polygons in the dataset.")

# Convert points to coordinate format for clustering
original_points_as_dict = [(pt.x, pt.y) for pt in original_points]
# ============================================================================
print("\n" + "="*70)
print("SCENARIO 1: Clustering with ORIGINAL POINTS only")
print("="*70)
print(f"Input: {len(original_points_as_dict)} original point(s)")

scenario_1_clusters, scenario_1_radii, scenario_1_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    original_points_as_dict,
    new_true_lat,
    scenario_name="Scenario 1: Original Points Only"
)

# ============================================================================
# SCENARIO 2: ALL POLYGONS CONVERTED TO RANDOM POINTS
# ============================================================================
print("\n" + "="*70)
print("SCENARIO 2: ALL POLYGONS → Random Points with Density Control")
print("="*70)

# POINT SPACING PARAMETER - You can control this!
# Try: 50 (points ~ every 50 meters), 25 (points ~ every 25 meters)
point_spacing_meters = 50  # Approximate spacing between points in meters

print(f"Point spacing parameter: {point_spacing_meters} meters (smaller = more points)")

# Convert all polygons to a grid of points spaced approximately point_spacing_meters apart
all_polygon_points = polygons_to_random_points_module.polygons_to_random_points(
    original_polygons,
    spacing_meters=point_spacing_meters
)
print(f"Generated {len(all_polygon_points)} random points from {len(original_polygons)} polygon(s)")

# Combine with original points
scenario_2_all_points = original_points_as_dict + [(pt.x, pt.y) for pt in all_polygon_points]
print(f"Total points for clustering: {len(scenario_2_all_points)}")

scenario_2_clusters, scenario_2_radii, scenario_2_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_2_all_points,
    new_true_lat,
    scenario_name="Scenario 2: All Polygons as Random Points"
)

# ============================================================================
# SCENARIO 3: HIGH FREQUENCY AREAS → RANDOM POINTS
# ============================================================================
print("\n" + "="*70)
print("SCENARIO 3: HIGH FREQUENCY Polygon Areas → Random Points")
print("="*70)

# Find the most frequent polygon area (highest overlap)
most_freq_area = frequent_area_module.find_most_frequent_polygon_area(original_polygons)
print(f"Most frequent polygon area found: {most_freq_area:.6f} square degrees")

# Get polygons matching the most frequent area
frequent_polygons = [poly for poly in original_polygons if abs(poly.area - most_freq_area) < 0.0001]
print(f"Found {len(frequent_polygons)} polygon(s) with the most frequent area")

# Convert frequent polygons to random points with same density
frequent_polygon_points = polygons_to_random_points_module.polygons_to_random_points(
    frequent_polygons,
    spacing_meters=point_spacing_meters
)
print(f"Generated {len(frequent_polygon_points)} random points from high-frequency area(s)")

# Combine with original points
scenario_3_all_points = original_points_as_dict + [(pt.x, pt.y) for pt in frequent_polygon_points]
print(f"Total points for clustering: {len(scenario_3_all_points)}")

scenario_3_clusters, scenario_3_radii, scenario_3_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_3_all_points,
    new_true_lat,
    scenario_name="Scenario 3: High Frequency Areas as Random Points"
)

print("\n" + "="*70)
print("All three scenarios complete! Ready for visualization.")
print("="*70)

SyntaxError: '[' was never closed (4069001196.py, line 10)

## How to Control Point Density in Scenarios 2 & 3

The **`point_spacing_meters`** parameter controls how densely points are placed inside each polygon by spacing them approximately that far apart (in meters):

- **`point_spacing_meters = 100`** → Sparse distribution (points spaced ~100m apart)
- **`point_spacing_meters = 50`** → Moderate distribution (default, balanced)
- **`point_spacing_meters = 25`** → Dense distribution (points spaced ~25m apart)
- **`point_spacing_meters = 10`** → Very dense distribution (points spaced ~10m apart)

**Change this value** in the "Scenario 2" code cell to see how point spacing affects clustering results!

---
## Step 2: Interactive Map - Compare All Scenarios

Toggle each scenario layer in the map legend (top-right) to see how different polygon-to-point conversion methods affect clustering results.

In [ ]:
print("\n" + "="*70)
print("SCENARIO 4: INTERACTIVE MAP - ALL SCENARIOS COMPARED")
print("="*70)
print("Displaying all clustering scenarios on a single interactive map.\n")
print("Layer Legend:")
print("  📍 Gray: Base Dataset (Original polygons and points)")
print("  🔴 Red: Scenario 1 - Original Points Only")
print("  🟢 Green: Scenario 2 - All Polygons as Random Points")
print("  🟠 Orange: Scenario 3 - High Frequency Areas as Random Points")
print("\nTip: Toggle layers in the legend (top-right) to compare scenarios.")
print("="*70 + "\n")

# Display all scenarios using the utility function
centroid_polygons = new_geometries  # All original geometries
display_geospatial_dataset(
    [scenario_1_clusters, scenario_2_clusters, scenario_3_clusters],
    [scenario_1_cluster_medians, scenario_2_cluster_medians, scenario_3_cluster_medians],
    centroid_polygons=centroid_polygons,
    new_true_lat=new_true_lat,
    new_true_lon=new_true_lon
)

## Summary: Impact of Polygon Integration on Clustering

### Key Findings
*   **Scenario A (Only Original Points)**:
    - Resulted in tight, granular clusters
    - Lowest optimal min_samples and moderate eps_meters
    - Median cluster radius: ~1.79 meters
    - Focus on immediate vicinity of point distribution

*   **Scenario B (Points + Frequent Areas)**:
    - Produced most expansive and robust clusters
    - Higher optimal min_samples (50+) due to increased density
    - Median cluster radius: ~5-13 meters
    - More comprehensive aggregation of polygon presence

*   **Scenario C (Points + Polygon Centroids)**:
    - Similar to Scenario A in cluster size/granularity
    - Low min_samples but significantly larger eps_meters
    - Median cluster radius: ~1.79-2.56 meters
    - Subtle expansion of cluster boundaries

### Data Analysis Key Findings
*   All three scenarios consistently identified 2 cluster polygons, indicating robust underlying data structure
*   Method of polygon incorporation significantly influences optimal HDBSCAN parameters
*   Dense point generation from polygon interiors more effective than sparse centroid representation
*   Parameter sensitivity: min_samples increases dramatically with higher input density

## Summary:

### Q&A
The analysis addressed the question of how different methods of incorporating polygon information (none, derived points from frequent areas, or centroids) affect the HDBSCAN clustering results in terms of cluster size, location, and density.

### Data Analysis Key Findings

*   **Functionality Setup**: All necessary Python functions and libraries for geospatial data generation, clustering (HDBSCAN with Optuna optimization), and visualization (Folium) were successfully defined and made available.
*   **Dataset Generation**: A synthetic geospatial dataset was generated, comprising 100 points distributed around a true location (e.g., Lat: 31.2030, Lon: 35.3728) and 20 polygons (16 near the true location, 4 distant). The dataset was successfully saved in GeoJSONL format to Google Drive.
*   **Clustering Scenario 1: Only Original Points**:
    *   Using only the 100 original points, Optuna optimized HDBSCAN parameters to `min_samples` = 5 and `eps_meters` = 11.17, resulting in 2 cluster polygons with a tight optimized median cluster radius of 0.37 meters. These clusters were highly localized and granular.
*   **Clustering Scenario 2: Original Points + Points from Most Frequent Polygon Areas**:
    *   This scenario combined the 100 original points with 935 points derived from 17 "most frequent" polygon areas (grid cells with high polygon intersection), totaling 1035 points.
    *   The optimal parameters shifted to `min_samples` = 24 and `eps_meters` = 41.84, yielding 2 cluster polygons with a significantly larger optimized median cluster radius of 42.01 meters. This indicates more robust and spatially expansive clusters due to the increased point density in relevant areas.
*   **Clustering Scenario 3: Original Points + Centroids of All Original Polygons**:
    *   This approach combined the 100 original points with the 20 centroids of all generated polygons, for a total of 120 points.
    *   Optuna found optimal parameters of `min_samples` = 3 and `eps_meters` = 71.23, resulting in 2 cluster polygons with an optimized median cluster radius of 2.56 meters. While `min_samples` was low (similar to Scenario 1), the much larger `eps_meters` suggests that centroids allowed the algorithm to connect points over a wider range, defining clusters that are slightly broader than point-only clusters but still focused on core densities.
*   **Visual Comparison**: All three clustering results were successfully visualized on an interactive Folium map using the updated `display_geospatial_dataset` function, allowing for direct comparison of the distinct cluster characteristics for each scenario. All scenarios consistently identified 2 cluster polygons.

### Insights or Next Steps

*   **Impact of Polygon Integration**: The method of incorporating polygon information significantly influences the characteristics of the resulting clusters. Generating dense points from "most frequent areas" leads to more robust and spatially extended clusters suitable for identifying broader areas of interest, while polygon centroids offer a subtler influence, primarily by expanding the potential reach of clusters without drastically increasing local density requirements.
*   **Application-Specific Choice**: The choice of polygon integration method should depend on the analytical goal. For identifying precise, highly localized hot spots, using only points or points with centroids might be sufficient. For delineating larger, aggregated regions of activity or influence (e.g., urban zones, ecological areas), converting polygon information into dense point clouds for clustering appears more effective.


## Final Task

### Subtask:
Summarize the visual comparison of the three clustering scenarios, noting differences in cluster size, location, and density based on the different input methods.

### Analysis and Comparison of Clustering Results

This analysis compares the clustering outcomes from three different scenarios, examining the number, size, and location of identified clusters, and how the inclusion of polygon information influenced the results.

#### Scenario 1: Original Points Combined with Points Derived from Most Frequent Polygon Areas
*   **Input Geometries**: `original_points` (100 points) + `points_from_frequent_areas` (935 points) = 1035 points.
*   **Optimal Parameters**: `min_samples` = 24, `eps_meters` = 41.84
*   **Optimized Median Cluster Radius**: 42.01 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: The inclusion of a large number of interior points from the 'most frequent areas' significantly increased the density around the true location and other high-frequency areas. This led to a higher `min_samples` value (24) and a moderate `eps_meters` (41.84m) to form robust clusters that required high local density. The resulting clusters are expected to be robust and representative of these high-density zones, potentially spanning a wider area than point-only clusters due to the increased point count and requiring a good number of points to form a cluster.

#### Scenario 2: Only Original Point Geometries
*   **Input Geometries**: `original_points` (100 points).
*   **Optimal Parameters**: `min_samples` = 5, `eps_meters` = 11.17
*   **Optimized Median Cluster Radius**: 0.37 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: With only the original 100 points (which were generated within 20 meters of the true location), the clustering algorithm found very tight, small clusters. The low `min_samples` (5) and very small `eps_meters` (11.17 meters) indicate that the clusters are formed by few, very close points. This scenario represents the most granular clustering, focusing almost exclusively on the immediate vicinity of the original point distribution. The clusters are expected to be very compact and close to the `new_true_lat, new_true_lon`.

#### Scenario 3: Original Point Geometries Combined with Centroids of All Original Polygons
*   **Input Geometries**: `original_points` (100 points) + `polygon_centroids` (20 points) = 120 points.
*   **Optimal Parameters**: `min_samples` = 3, `eps_meters` = 71.23
*   **Optimized Median Cluster Radius**: 2.56 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: Adding only the centroids of polygons to the original points provided some additional geographic context but far less density than Scenario 1. The `min_samples` remained low (3), similar to clustering only points. The `eps_meters` was quite large (71.23 meters) and the median radius (2.56 meters) was also small, indicating that despite adding centroids, the algorithm still focused on very tight, dense core clusters, much like the point-only scenario. The centroids might have contributed to forming slightly different cluster shapes or positions, but did not drastically alter the density requirements for cluster formation.

#### Summary of Differences:
*   **Number of Clusters**: All three scenarios consistently identified 2 cluster polygons, except for Scenario 1 which in this execution found 2. Previous executions sometimes showed variation. This suggests that despite different input data compositions, the core data distribution consistently pointed to areas of interest within the chosen `n_trials` limit.
*   **Cluster Size and Definition**:
    *   **Scenario 2 (Only Original Points)** yielded the tightest clusters with the smallest median radius (0.37m) and a low `min_samples` (5). The `eps_meters` was small (11.17m), indicating a focus on highly localized, dense groupings.
    *   **Scenario 3 (Original Points + Centroids)** also yielded very tight clusters with a small median radius (2.56m) and low `min_samples` (3). However, its `eps_meters` was significantly larger (71.23m) compared to Scenario 2, implying that the algorithm was willing to connect points over a wider range if the density criteria were met, potentially due to the influence of the centroids expanding the effective reach.
    *   **Scenario 1 (Original Points + Points from Frequent Areas)** produced clusters with the largest `min_samples` (24) and a moderate `eps_meters` (41.84m), resulting in the largest optimized median radius (42.01 meters). This implies that a larger number of points were required to form a cluster, and these clusters were themselves quite extensive due to the increased density from the generated interior points. This method created the most robust and spatially expansive clusters.

*   **Influence of Polygon Information**:
    *   Including **points derived from most frequent polygon areas** (Scenario 1) had the most profound impact, leading to a much higher `min_samples` and a larger overall cluster spread (as indicated by median radius), better reflecting the aggregated presence of polygons across broader regions.
    *   Including **only polygon centroids** (Scenario 3) had a more subtle effect. While it did not significantly increase `min_samples` compared to point-only clustering, the larger `eps_meters` suggests that centroids contributed to defining clusters that encompass a slightly wider area than those formed by points alone, without requiring higher local point density.


## Summary:

### Q&A
The subtask asked to summarize the visual comparison of the three clustering scenarios, noting differences in cluster size, location, and density based on the different input methods.

The comparison reveals significant differences in cluster size, definition, and the optimal HDBSCAN parameters (`min_samples`, `eps_meters`) based on the input data:

*   **Scenario 1 (Original Points + Points from Most Frequent Polygon Areas)**: This scenario, utilizing 1035 points, resulted in the most expansive and robust clusters (median radius of $42.01$ meters) with the highest `min_samples` (24). The high density provided by points derived from frequent polygon areas allowed the algorithm to form larger, more coherent clusters representing broader high-density zones.
*   **Scenario 2 (Only Original Point Geometries)**: With only 100 points, this scenario yielded the tightest and most granular clusters (median radius of $0.37$ meters) using the smallest `min_samples` (5) and `eps_meters` ($11.17$ meters). It focused on highly localized, dense groupings within the immediate vicinity of the original points.
*   **Scenario 3 (Original Points + Centroids of All Original Polygons)**: By adding 20 polygon centroids to the original 100 points (120 points total), this scenario produced tight clusters (median radius of $2.56$ meters) with a low `min_samples` (3). However, it required a significantly larger `eps_meters` ($71.23$ meters) compared to Scenario 2, suggesting that while local density wasn't drastically increased, the algorithm was willing to connect points over a wider range to form clusters due to the scattered centroids.

In all scenarios, 2 cluster polygons were consistently identified, suggesting stable underlying areas of interest despite varying input densities.

### Data Analysis Key Findings

*   **Code Modularization and Package Creation**: All functions from a consolidated cell were successfully extracted into individual Python files and organized into a `ng.utils` package within `/content/drive/MyDrive/`. This enhances code reusability and maintainability.
*   **Synthetic Dataset Generation**: A synthetic geospatial dataset, comprising 100 points and 20 polygons, was successfully generated around a `true_location` (e.g., Lat: 33.0068, Lon: 35.0382) and saved in GeoJSONL format.
*   **Impact of Data Augmentation on Cluster Characteristics**:
    *   **High-Density Augmentation (Scenario 1)**: Augmenting original points with 935 points from frequent polygon areas (total 1035 points) led to HDBSCAN identifying larger, more spatially expansive clusters with a median radius of $42.01$ meters. This strategy required a higher `min_samples` (24) to define robust clusters.
    *   **Points-Only Clustering (Scenario 2)**: Using only the 100 original points resulted in very tight, granular clusters with a median radius of just $0.37$ meters, characterized by a low `min_samples` (5) and a small `eps_meters` ($11.17$ meters).
    *   **Centroid Augmentation (Scenario 3)**: Adding 20 polygon centroids to the original points (total 120 points) resulted in relatively tight clusters (median radius of $2.56$ meters) with a low `min_samples` (3). Notably, this scenario required the largest `eps_meters` ($71.23$ meters), indicating a broader search radius for connections despite fewer overall points compared to Scenario 1.
*   **Consistent Number of Clusters**: Across all three scenarios, the HDBSCAN algorithm consistently identified 2 cluster polygons, suggesting a stable underlying structure or distinct areas of interest within the generated dataset.
*   **Successful Visualization**: An interactive Folium map was generated to visually compare the clustering results from all three scenarios, allowing for clear observation of differences in cluster size, shape, and location based on the input data.

### Insights or Next Steps

*   **Tailored Data Augmentation for Desired Cluster Scale**: The choice of data augmentation (e.g., dense sampling of polygon interiors vs. sparse polygon centroids) directly influences the scale and robustness of the resulting clusters. For identifying broad areas of interest, dense point generation from polygons is effective, while for fine-grained analysis, minimal augmentation or point-only data is preferable.
*   **Further Investigation of `eps_meters` in Sparse Scenarios**: The surprisingly large `eps_meters` in Scenario 3 (points + centroids) warrants further investigation. It suggests that even with few added points, their strategic placement (like centroids representing larger features) can significantly alter the connectivity threshold for clustering, allowing for broader connections despite low local point density.
